In [1]:
import os
import numpy as np
import pandas as pd

os.environ.setdefault("PYTHONHASHSEED", "0")
np.random.seed(42)

base_dir = "/kaggle/input/jigsaw-unintended-bias-in-toxicity-classification"
fallback_dir = "/kaggle/input"


def pick_path(filename: str) -> str:
    p1 = os.path.join(base_dir, filename)
    if os.path.exists(p1):
        return p1
    p2 = os.path.join(fallback_dir, filename)
    if os.path.exists(p2):
        return p2
    raise FileNotFoundError(
        f"Could not find {filename} in {base_dir} or {fallback_dir}"
    )


train_path = pick_path("train.csv")
test_path = pick_path("test.csv")
sample_path = pick_path("sample_submission.csv")

sample_sub = pd.read_csv(sample_path, usecols=["id", "prediction"])
required_cols = ["id", "prediction"]
missing = [c for c in required_cols if c not in sample_sub.columns]
if missing:
    raise ValueError(f"sample_submission is missing required columns: {missing}")
sample_sub = sample_sub[required_cols].copy()

print("Paths:", train_path, test_path, sample_path)



Paths: /kaggle/input/jigsaw-unintended-bias-in-toxicity-classification/train.csv /kaggle/input/jigsaw-unintended-bias-in-toxicity-classification/test.csv /kaggle/input/jigsaw-unintended-bias-in-toxicity-classification/sample_submission.csv


In [2]:
from typing import Iterator, Tuple, List

TRAIN_CHUNK_ROWS = 200_000
TEST_CHUNK_ROWS = 200_000

TRAIN_USECOLS = ["comment_text", "target"]
TEST_USECOLS = ["id", "comment_text"]

TRAIN_DTYPES = {"comment_text": "object", "target": "float32"}
TEST_DTYPES = {"id": "int64", "comment_text": "object"}


def iter_train_chunks() -> Iterator[Tuple[np.ndarray, np.ndarray]]:
    reader = pd.read_csv(
        train_path,
        usecols=TRAIN_USECOLS,
        dtype=TRAIN_DTYPES,
        chunksize=TRAIN_CHUNK_ROWS,
    )
    for df in reader:
        txt = df["comment_text"].fillna("").to_numpy()
        y = (df["target"].to_numpy(copy=False) >= 0.5).astype(np.int32, copy=False)
        yield txt, y


def iter_test_chunks() -> Iterator[Tuple[np.ndarray, np.ndarray]]:
    reader = pd.read_csv(
        test_path,
        usecols=TEST_USECOLS,
        dtype=TEST_DTYPES,
        chunksize=TEST_CHUNK_ROWS,
    )
    for df in reader:
        ids = df["id"].to_numpy(copy=False)
        txt = df["comment_text"].fillna("").to_numpy()
        yield ids, txt


def cache_train_chunks() -> Tuple[List[np.ndarray], List[np.ndarray], int, int]:
    txt_chunks: List[np.ndarray] = []
    y_chunks: List[np.ndarray] = []
    n_docs = 0
    pos = 0
    for txt, y in iter_train_chunks():
        txt_chunks.append(txt)
        y_chunks.append(y)
        n_docs += len(y)
        pos += int(y.sum())
    return txt_chunks, y_chunks, n_docs, pos


train_txt_chunks, train_y_chunks, _n_docs_quick, _pos_quick = cache_train_chunks()
print(
    "Cached train chunks:",
    len(train_txt_chunks),
    "docs:",
    _n_docs_quick,
    "pos_rate:",
    _pos_quick / max(_n_docs_quick, 1),
)



Cached train chunks: 10 docs: 1804874 pos_rate: 0.0799690172277954


In [3]:
from sklearn.feature_extraction.text import HashingVectorizer

N_WORD = 2**20  # 1,048,576 hashed word features
N_CHAR = 2**20  # 1,048,576 hashed char features
N_TOTAL = N_WORD + N_CHAR

word_hasher = HashingVectorizer(
    strip_accents="unicode",
    lowercase=True,
    analyzer="word",
    ngram_range=(1, 2),
    n_features=N_WORD,
    alternate_sign=False,
    norm=None,
)

char_hasher = HashingVectorizer(
    strip_accents="unicode",
    lowercase=True,
    analyzer="char",
    ngram_range=(3, 5),
    n_features=N_CHAR,
    alternate_sign=False,
    norm=None,
)

print("HashingVectorizers ready. Total features:", N_TOTAL)



HashingVectorizers ready. Total features: 2097152


In [4]:
import numpy as np
from scipy import sparse as sp


def combine_word_char_counts(Xw: sp.csr_matrix, Xc: sp.csr_matrix) -> sp.csr_matrix:
    Xw = Xw.tocsr(copy=False)
    Xc = Xc.tocsr(copy=False)
    X = sp.hstack((Xw, Xc), format="csr", dtype=np.float64)
    if not X.has_sorted_indices:
        X.sort_indices()
    return X


# CHANGE (timeout fix): Precompute and cache hashed COUNT matrices once per training chunk.
# This preserves core logic (same hashed features and counts) but avoids doing the expensive
# HashingVectorizer.transform + hstack 100 times (once per epoch), which caused the timeout.
def cache_train_count_matrices() -> tuple[list[sp.csr_matrix], int]:
    X_counts_chunks: list[sp.csr_matrix] = []
    n_docs = 0
    print("Caching hashed count matrices for train chunks (one-time)...")
    for i, txt in enumerate(train_txt_chunks):
        Xw = word_hasher.transform(txt)
        Xc = char_hasher.transform(txt)
        X_counts = combine_word_char_counts(Xw, Xc)  # CSR float64 counts
        X_counts_chunks.append(X_counts)
        n_docs += X_counts.shape[0]
        if (i + 1) % 5 == 0:
            print(f"Cached X_counts chunks={i+1}, docs={n_docs}")
    return X_counts_chunks, n_docs


train_X_counts_chunks, n_docs = cache_train_count_matrices()


# CHANGE (timeout fix): Compute DF using CSR structure without allocating X.copy()/binarized matrix.
# For each row, each column index appears at most once in CSR from HashingVectorizer output, so
# counting unique columns per row equals document frequency. This is exactly equivalent to summing
# a binarized copy, but avoids a massive extra sparse allocation and data fill.
def compute_df_from_cached_counts(X_counts_chunks: list[sp.csr_matrix]) -> np.ndarray:
    df_counts = np.zeros(N_TOTAL, dtype=np.int64)
    print("Computing DF/IDF from cached count matrices using indptr-based counting...")
    for chunk_i, X in enumerate(X_counts_chunks):
        # For each nonzero entry (i,j), add 1 to df[j] exactly once per document.
        # CSR from vectorizers has unique indices per row; no duplicates to collapse.
        row_nnz = np.diff(X.indptr).astype(np.int64, copy=False)  # nnz per row
        df_counts += np.bincount(
            X.indices, weights=np.repeat(1, X.indices.shape[0]), minlength=N_TOTAL
        )
        if (chunk_i + 1) % 5 == 0:
            docs_so_far = sum(mat.shape[0] for mat in X_counts_chunks[: chunk_i + 1])
            print(f"DF pass: processed chunks={chunk_i+1}, docs~={docs_so_far}")
    return df_counts


# NOTE: np.bincount with weights=np.repeat(1, nnz) is equivalent to counting occurrences.
# Kept explicit for clarity; avoids creating a full binarized sparse matrix.
df_counts = compute_df_from_cached_counts(train_X_counts_chunks)

idf = (np.log((1.0 + n_docs) / (1.0 + df_counts.astype(np.float64))) + 1.0).astype(
    np.float64, copy=False
)

print(
    "Computed IDF. n_docs:",
    n_docs,
    "idf stats:",
    float(idf.min()),
    float(idf.mean()),
    float(idf.max()),
)



Caching hashed count matrices for train chunks (one-time)...
Cached X_counts chunks=5, docs=1000000
Cached X_counts chunks=10, docs=1804874
Computing DF/IDF from cached count matrices using indptr-based counting...


UFuncTypeError: Cannot cast ufunc 'add' output from dtype('float64') to dtype('int64') with casting rule 'same_kind'

In [5]:
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import normalize
from scipy.sparse import csr_matrix
import numpy as np

C = 4.0
alpha = 1.0 / (C * float(n_docs))

clf = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=alpha,
    fit_intercept=True,
    max_iter=1,  # one pass per epoch; we control epochs explicitly
    tol=None,
    shuffle=False,  # determinism + consistent with streaming
    random_state=42,
    learning_rate="optimal",
    average=False,
)

EPOCHS = 100
classes = np.array([0, 1], dtype=np.int32)


def tfidf_from_counts_inplace(X_counts: csr_matrix) -> csr_matrix:
    X = X_counts
    X.data = np.log(X.data) + 1.0
    X = X.multiply(idf)
    return normalize(X, norm="l2", copy=False)


# CHANGE (timeout fix): Reuse cached count matrices per chunk for every epoch.
# TF-IDF transformation is still applied per-epoch (same as before), but the expensive
# hashing/vectorization step is removed from the inner training loop.
for epoch in range(EPOCHS):
    for batch_i, (X_counts_cached, yb) in enumerate(
        zip(train_X_counts_chunks, train_y_chunks)
    ):
        # Need a copy because tfidf_from_counts_inplace mutates X.data; reuse counts across epochs.
        X_tfidf = tfidf_from_counts_inplace(X_counts_cached.copy())

        if epoch == 0 and batch_i == 0:
            clf.partial_fit(X_tfidf, yb, classes=classes)
        else:
            clf.partial_fit(X_tfidf, yb)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} done")

print("Training done.")



NameError: name 'idf' is not defined

In [6]:
import numpy as np
import pandas as pd

test_n = int(sample_sub.shape[0])
test_ids = np.empty(test_n, dtype=np.int64)
pred = np.empty(test_n, dtype=np.float64)

off = 0
for ids, txt in iter_test_chunks():
    Xw = word_hasher.transform(txt)
    Xc = char_hasher.transform(txt)
    X_counts = combine_word_char_counts(Xw, Xc)
    X_tfidf = tfidf_from_counts_inplace(X_counts)

    p = clf.predict_proba(X_tfidf)[:, 1].astype(np.float64, copy=False)
    p = np.clip(p, 0.0, 1.0)

    n = len(ids)
    test_ids[off : off + n] = ids
    pred[off : off + n] = p
    off += n

if off != test_n:
    test_ids = test_ids[:off]
    pred = pred[:off]

print(
    "Pred stats:",
    float(pred.min()),
    float(pred.mean()),
    float(pred.max()),
    "n_test:",
    pred.shape[0],
)

pred_df = pd.DataFrame({"id": test_ids, "prediction": pred})
res = sample_sub[["id"]].merge(pred_df, on="id", how="left")
res["prediction"] = (
    pd.to_numeric(res["prediction"], errors="coerce").fillna(0.0).clip(0.0, 1.0)
)

out_path = "submission.csv"
res.to_csv(out_path, index=False)

print(f"Wrote {out_path} with shape={res.shape} and columns={list(res.columns)}")
print(res.head())
print("Missing predictions:", int(res["prediction"].isna().sum()))

NameError: name 'idf' is not defined